In [ ]:
### LSTM과 GRU 모델


## IMDB 리뷰 데이터셋 호출 & 추출

from keras.datasets import imdb
from sklearn.model_selection import train_test_split
from keras.preprocessing.sequence import pad_sequences

(train_input, train_target), (test_input, test_target) = imdb.load_data(num_words = 500)

# 훈련-검증 세트 추출
train_input, val_input, train_target, val_target = train_test_split(train_input, train_target, test_size = 0.2, random_state = 42)

# 훈련-검증 샘플 패딩(길이 통일)
train_seq = pad_sequences(train_input, maxlen = 100)
val_seq = pad_sequences(val_input, maxlen = 100)



## LSTM 모델 구현

import keras

model_lstm = keras.Sequential()
model_lstm.add(keras.layers.Input(shape = (100,)))
model_lstm.add(keras.layers.Embedding(500, 100)) # 임베딩 층으로 단어 표현 - 용량 효율 + 단어 관계 반영
model_lstm.add(keras.layers.LSTM(8))
model_lstm.add(keras.layers.Dense(1, activation = 'sigmoid')) # 출력층 - 시그모이드 활성화 함수

model_lstm.summary()



## LSTM 모델 컴파일 & 훈련
model_lstm.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
checkpoint_cb = keras.callbacks.ModelCheckpoint('best-lstm-model.keras', save_best_only = True)
early_stopping_cb = keras.callbacks.EarlyStopping(patience = 3, restore_best_weights = True)
history = model_lstm.fit(train_seq, train_target, epochs= 100, batch_size = 64, validation_data=(val_seq, val_target), callbacks = [checkpoint_cb, early_stopping_cb])


# LSTM 모델의 훈련 & 검증 손실 시각화 및 관찰
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label = 'train')
plt.plot(history.history['val_loss'], label = 'val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()



## 드롭아웃을 적용한 LSTM 모델 구현 & 훈련

model_dropout = keras.Sequential()
model_dropout.add(keras.layers.Input(shape = (100,)))
model_dropout.add(keras.layers.Embedding(500, 100))
model_dropout.add(keras.layers.LSTM(8, dropout = 0.2))
model_dropout.add(keras.layers.Dense(1, activation = 'sigmoid'))
# 드롭아웃 추가를 제외하고서는 초기 LSTM 모델과 동일

model_dropout.summary()

model_dropout.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
checkpoint_cb = keras.callbacks.ModelCheckpoint('best-dropout-model.keras', save_best_only = True)
early_stopping_cb = keras.callbacks.EarlyStopping(patience = 3, restore_best_weights = True)
history = model_dropout.fit(train_seq, train_target, epochs= 100, batch_size = 64, validation_data=(val_seq, val_target), callbacks = [checkpoint_cb, early_stopping_cb])


# 관찰 결과 - 드롭아웃 적용 시 미미한 성능 향상
plt.plot(history.history['loss'], label = 'train')
plt.plot(history.history['val_loss'], label = 'val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()



## 2개 이상의 층 연결하기

model_2lstm = keras.Sequential()
model_2lstm.add(keras.layers.Input(shape = (100,)))
model_2lstm.add(keras.layers.Embedding(500, 100))
# LSTM 층의 return_sequences = True -> 다음 LSTM 층에 모든 은닉 상태를 전달(순차 데이터 보존)
model_2lstm.add(keras.layers.LSTM(8, dropout = 0.2, return_sequences= True))
model_2lstm.add(keras.layers.LSTM(8, dropout = 0.2))
model_2lstm.add(keras.layers.Dense(1, activation = 'sigmoid'))

model_2lstm.summary()

model_2lstm.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
checkpoint_cb = keras.callbacks.ModelCheckpoint('best-2lstm-model.keras', save_best_only = True)
early_stopping_cb = keras.callbacks.EarlyStopping(patience = 3, restore_best_weights = True)
history = model_2lstm.fit(train_seq, train_target, epochs= 100, batch_size = 64, validation_data=(val_seq, val_target), callbacks = [checkpoint_cb, early_stopping_cb])


plt.plot(history.history['loss'], label = 'train')
plt.plot(history.history['val_loss'], label = 'val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()



## GRU 모델 구현 + 훈련 진행

model_gru = keras.Sequential()
model_gru.add(keras.layers.Input(shape = (100,)))
model_gru.add(keras.layers.Embedding(500, 100))
model_gru.add(keras.layers.GRU(8, dropout = 0.2))
model_gru.add(keras.layers.Dense(1, activation = 'sigmoid'))
# LSTM의 간소화 버전인 GRU 모델 -> 구조 + 파라미터 매우 유사

model_gru.summary()

model_gru.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
checkpoint_cb = keras.callbacks.ModelCheckpoint('best-gru-model.keras', save_best_only = True)
early_stopping_cb = keras.callbacks.EarlyStopping(patience = 3, restore_best_weights = True)
history = model_gru.fit(train_seq, train_target, epochs= 100, batch_size = 64, validation_data=(val_seq, val_target), callbacks = [checkpoint_cb, early_stopping_cb])


# 관찰 결과 - GRU -> LSTM보다 적은 파라미터를 창출하면서도 유사한 성능(효율성)
plt.plot(history.history['loss'], label = 'train')
plt.plot(history.history['val_loss'], label = 'val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()


## 테스트 세트 평가(마지막 단 1회)

test_seq = pad_sequences(test_input, maxlen = 100)
model = keras.models.load_model('best-gru-model.keras')
# ModelCheckpoint로 저장한 최적 GRU 모델 구조 불러오기
model.evaluate(test_seq, test_target) # 최종 테스트 세트 평가 - 검증 정확도에 근접